In [ ]:
import asyncio
import os
import json
import hashlib
import logging
import re
import time
from typing import List, Tuple, Optional
from firecrawl import FirecrawlApp
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('apollo_podcast_extractor.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Constants
CACHE_DIR = "apollo_podcast"
os.makedirs(CACHE_DIR, exist_ok=True)


In [ ]:
# Configuration
FIRECRAWL_API_KEY = ""  # Add your Firecrawl API key here
CREDENTIALS_FILE = '/Users/utkarshumang/Downloads/url-to-email-445616-cebe4868914f.json'  # Path to your Google Sheets service account JSON file
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1Leh0yMDHuU59XOkRyxGZF4lGufGReHOC85k3bVuX-4I/edit?gid=572755901#gid=572755901"  # Google Sheets URL


In [ ]:
class GoogleSheetsManager:
    """Manages Google Sheets operations to read URLs from column Y"""
    
    def __init__(self, credentials_file: str):
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(
            credentials_file, scopes=scopes
        )
        self.service = build('sheets', 'v4', credentials=creds)
    
    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        """Extract spreadsheet ID from Google Sheets URL"""
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError("Invalid Google Sheet URL")
    
    def get_urls(self, spreadsheet_id: str, start_row: int = 2) -> List[Tuple[int, str]]:
        """Get URLs from column Y starting from specified row"""
        range_name = f"Y{start_row}:Y"  # Column Y for URLs
        result = self.service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id, 
            range=range_name
        ).execute()
        values = result.get('values', [])
        return [
            (i + start_row, row[0]) 
            for i, row in enumerate(values) 
            if row and row[0].strip()
        ]


In [ ]:
class FirecrawlWrapper:
    """Wrapper for Firecrawl API with caching to apollo_podcast folder"""
    
    def __init__(self, api_key: str):
        self.app = FirecrawlApp(api_key=api_key)
    
    def _hash_url(self, url: str) -> str:
        """Generate MD5 hash of URL for filename"""
        return hashlib.md5(url.encode()).hexdigest()
    
    def _get_cache_path(self, url: str) -> str:
        """Get cache file path for a URL"""
        return os.path.join(CACHE_DIR, f"{self._hash_url(url)}.json")
    
    def get_cached_result(self, url: str) -> Optional[List[str]]:
        """Check if URL is cached and return cached result"""
        cache_path = self._get_cache_path(url)
        if os.path.exists(cache_path):
            try:
                with open(cache_path, 'r') as f:
                    links = json.load(f)
                    logger.info(f"Cache hit for {url}")
                    return links
            except Exception as e:
                logger.warning(f"Error reading cache for {url}: {e}")
                return None
        return None
    
    def map_url(self, url: str) -> List[str]:
        """Map URL using Firecrawl API and cache the result"""
        # Check cache first
        cached = self.get_cached_result(url)
        if cached:
            return cached
        
        # Call Firecrawl API
        try:
            result = self.app.map_url(url)
            if getattr(result, 'success', False):
                links = result.links
                # Save to cache
                cache_path = self._get_cache_path(url)
                with open(cache_path, 'w') as f:
                    json.dump(links, f, indent=2)
                logger.info(f"Saved {len(links)} URLs for {url}")
                return links
            else:
                logger.warning(f"Firecrawl returned unsuccessful result for {url}")
                return []
        except Exception as e:
            logger.error(f"Firecrawl error for {url}: {e}")
            return []


In [ ]:
async def process_url(
    url: str, 
    row_num: int,
    firecrawl: FirecrawlWrapper,
    semaphore: asyncio.Semaphore,
    stats: dict
) -> Tuple[int, str, List[str]]:
    """Process a single URL with Firecrawl map endpoint"""
    async with semaphore:
        try:
            # Check cache first (without logging)
            cache_path = firecrawl._get_cache_path(url)
            was_cached = os.path.exists(cache_path)
            
            # Run Firecrawl map_url in executor since it's synchronous
            # map_url will use cache if available, or make API call if not
            loop = asyncio.get_event_loop()
            links = await loop.run_in_executor(None, firecrawl.map_url, url)
            
            stats['processed'] += 1
            if was_cached:
                stats['cache_hit'] += 1
            else:
                stats['api_calls'] += 1
            
            if links:
                stats['total_urls_found'] += len(links)
            
            return (row_num, url, links)
        except Exception as e:
            stats['errors'] += 1
            logger.error(f"Error processing row {row_num} ({url}): {e}")
            return (row_num, url, [])


async def process_all_urls(
    sheet_url: str,
    firecrawl_api_key: str,
    credentials_file: str,
    start_row: int = 2
):
    """Main function to process all URLs from Google Sheet"""
    logger.info("Starting Apollo Podcast URL extraction")
    
    # Initialize managers
    sheet_mgr = GoogleSheetsManager(credentials_file)
    firecrawl = FirecrawlWrapper(firecrawl_api_key)
    
    # Get spreadsheet ID and URLs
    spreadsheet_id = sheet_mgr.extract_spreadsheet_id(sheet_url)
    urls = sheet_mgr.get_urls(spreadsheet_id, start_row)
    
    logger.info(f"Found {len(urls)} URLs to process")
    
    # Statistics tracking
    stats = {
        'processed': 0,
        'cache_hit': 0,
        'api_calls': 0,
        'total_urls_found': 0,
        'errors': 0
    }
    
    # Create semaphore for 50 concurrent requests
    semaphore = asyncio.Semaphore(50)
    
    # Process all URLs concurrently
    tasks = [
        process_url(url, row_num, firecrawl, semaphore, stats)
        for row_num, url in urls
    ]
    
    logger.info(f"Processing {len(tasks)} URLs with 50 concurrent requests...")
    start_time = time.time()
    
    # Wait for all tasks to complete
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    elapsed_time = time.time() - start_time
    
    # Log final statistics
    logger.info("=" * 60)
    logger.info("Processing Complete!")
    logger.info(f"Total URLs processed: {stats['processed']}")
    logger.info(f"Cache hits: {stats['cache_hit']}")
    logger.info(f"API calls made: {stats['api_calls']}")
    logger.info(f"Total sub-URLs discovered: {stats['total_urls_found']}")
    logger.info(f"Errors encountered: {stats['errors']}")
    logger.info(f"Time elapsed: {elapsed_time:.2f} seconds")
    logger.info(f"Results saved to: {CACHE_DIR}/")
    logger.info("=" * 60)
    
    return results


In [ ]:
# Execute the pipeline
await process_all_urls(
    sheet_url=GOOGLE_SHEET_URL,
    firecrawl_api_key=FIRECRAWL_API_KEY,
    credentials_file=CREDENTIALS_FILE,
    start_row=27650
)
